In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
np.random.seed(42)

print("Libraries loaded!")

## 12.1 Error Analysis Process

In [ ]:
# Load digits
digits = load_digits()
X = digits.data
y = digits.target

# Split: 60% train, 20% dev, 20% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_dev, X_test, y_dev, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {len(X_train)} samples")
print(f"Dev: {len(X_dev)} samples")
print(f"Test: {len(X_test)} samples")

# Train baseline model
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)

# Evaluate
train_acc = clf.score(X_train, y_train)
dev_acc = clf.score(X_dev, y_dev)
test_acc = clf.score(X_test, y_test)

print(f"\nBaseline Performance:")
print(f"Train accuracy: {train_acc:.3f}")
print(f"Dev accuracy: {dev_acc:.3f}")
print(f"Test accuracy: {test_acc:.3f}")

# Error analysis on dev set
y_pred_dev = clf.predict(X_dev)
errors = y_pred_dev != y_dev
error_indices = np.where(errors)[0]

print(f"\nTotal errors on dev set: {np.sum(errors)}")

# Visualize errors
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    if i < len(error_indices):
        idx = error_indices[i]
        ax.imshow(X_dev[idx].reshape(8, 8), cmap='gray')
        ax.set_title(f'True: {y_dev[idx]}\nPred: {y_pred_dev[idx]}', 
                    fontsize=9, color='red')
    ax.axis('off')

plt.suptitle('Error Analysis: Misclassified Digits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 12.2 Confusion Matrix Analysis

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_dev, y_pred_dev)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.title('Confusion Matrix - Dev Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Find most confused pairs
off_diagonal = cm.copy()
np.fill_diagonal(off_diagonal, 0)
confused_pairs = []

for i in range(10):
    for j in range(10):
        if off_diagonal[i, j] > 0:
            confused_pairs.append((i, j, off_diagonal[i, j]))

confused_pairs.sort(key=lambda x: x[2], reverse=True)

print("\nMost Confused Digit Pairs:")
for true_label, pred_label, count in confused_pairs[:5]:
    print(f"  {true_label} → {pred_label}: {count} errors")

## 12.3 Diagnostic Decisions

In [ ]:
# Decision tree for debugging
print("ML Debugging Decision Tree:\n")

train_error = 1 - train_acc
dev_error = 1 - dev_acc
gap = dev_error - train_error

print(f"Train error: {train_error:.3f}")
print(f"Dev error: {dev_error:.3f}")
print(f"Gap: {gap:.3f}\n")

# Diagnosis
if train_error > 0.1:  # High train error
    print("DIAGNOSIS: High Bias (Underfitting)")
    print("RECOMMENDATIONS:")
    print("  1. Try more complex model")
    print("  2. Add more features")
    print("  3. Train longer")
    print("  4. Reduce regularization")
elif gap > 0.05:  # Large train-dev gap
    print("DIAGNOSIS: High Variance (Overfitting)")
    print("RECOMMENDATIONS:")
    print("  1. Get more training data")
    print("  2. Regularization (L2, dropout)")
    print("  3. Reduce model complexity")
    print("  4. Data augmentation")
else:
    print("DIAGNOSIS: Good Fit")
    print("RECOMMENDATIONS:")
    print("  1. Error analysis for remaining errors")
    print("  2. Collect specific data for hard cases")
    print("  3. Ensemble methods")

## 12.4 Iterative Improvement

In [ ]:
# Simulate iterative improvement
iterations = ['Baseline', '+ More Trees', '+ Max Depth', '+ Min Samples']
train_accs = [train_acc]
dev_accs = [dev_acc]

# Iteration 1: More trees
clf1 = RandomForestClassifier(n_estimators=100, random_state=42)
clf1.fit(X_train, y_train)
train_accs.append(clf1.score(X_train, y_train))
dev_accs.append(clf1.score(X_dev, y_dev))

# Iteration 2: Tune max_depth
clf2 = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
clf2.fit(X_train, y_train)
train_accs.append(clf2.score(X_train, y_train))
dev_accs.append(clf2.score(X_dev, y_dev))

# Iteration 3: Tune min_samples_split
clf3 = RandomForestClassifier(n_estimators=100, max_depth=15, min_samples_split=5, random_state=42)
clf3.fit(X_train, y_train)
train_accs.append(clf3.score(X_train, y_train))
dev_accs.append(clf3.score(X_dev, y_dev))

# Plot progress
x = np.arange(len(iterations))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 7))
ax.bar(x - width/2, train_accs, width, label='Train Accuracy', alpha=0.8)
ax.bar(x + width/2, dev_accs, width, label='Dev Accuracy', alpha=0.8)

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Iterative Model Improvement', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(iterations, rotation=15, ha='right')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nProgress Summary:")
for i, iter_name in enumerate(iterations):
    print(f"{iter_name}: Train={train_accs[i]:.3f}, Dev={dev_accs[i]:.3f}, Gap={train_accs[i]-dev_accs[i]:.3f}")

## 12.5 Final Evaluation

In [ ]:
# Use best model on test set (ONLY ONCE!)
best_model = clf3
test_acc_final = best_model.score(X_test, y_test)
y_pred_test = best_model.predict(X_test)

print("FINAL TEST SET EVALUATION:")
print(f"Test Accuracy: {test_acc_final:.3f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_test))

print("\n" + "="*60)
print("IMPORTANT: Test set used only once for final evaluation!")
print("Never tune hyperparameters on test set.")
print("="*60)

## Key Takeaways

### 1. **Data Splits**
- **Train**: Fit model parameters
- **Dev/Validation**: Tune hyperparameters, select model
- **Test**: Final evaluation ONLY (never touch during development)
- Typical split: 60/20/20 or 80/10/10

### 2. **Error Analysis**
- Manually examine dev set errors
- Categorize error types
- Prioritize by frequency
- Find patterns (e.g., 8s confused with 3s)

### 3. **Debugging Checklist**
1. **High train error?** → Bigger model, more features
2. **Large train-dev gap?** → More data, regularization
3. **Large dev-test gap?** → Better dev set
4. **High dev error?** → Error analysis

### 4. **Iterative Process**
1. Start with simple baseline
2. Diagnose bottleneck (bias vs variance)
3. Try focused improvement
4. Evaluate on dev set
5. Repeat

### 5. **Common Mistakes**
- Tuning on test set
- Optimizing wrong metric
- Not doing error analysis
- Premature optimization
- Ignoring train-dev gap

### 6. **Best Practices**
- Single number evaluation metric
- Establish baseline quickly
- Iterate rapidly
- Document experiments
- Test set is sacred

## References

1. Andrew Ng's "Machine Learning Yearning"
2. "Deep Learning" - Goodfellow et al., Chapter 11
3. CS229 Lecture Notes

---

**Next**: [Lecture 13: Recommender Systems](13_recommender_systems.ipynb)